In [ ]:
!nvidia-smi

In [ ]:
# =========================
# CELL 1A — MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# CELL 1B — IMPORTS + CONFIG
# =========================
import os, json, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0

# ── EfficientNetB0: TIDAK pakai preprocess_input external ──
# Normalisasi dilakukan internal oleh model (input: float32 0-255)

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# PATH CONFIG — PROJECT
# =========================
PROJECT_DIR = "/content/drive/MyDrive/S-Class/Orion/OrionFL/orion-fl-dr-research"

DATASET_NAME    = "MultiClient"
EXPERIMENT_NAME = "exp1_centralized"
MODEL_NAME      = "efficientnetb0"

BASE_RESULT_DIR = os.path.join(PROJECT_DIR, "results", DATASET_NAME, EXPERIMENT_NAME, MODEL_NAME)
MODELS_DIR      = os.path.join(BASE_RESULT_DIR, "models")
LOGS_DIR        = os.path.join(BASE_RESULT_DIR, "logs")
FIGURES_DIR     = os.path.join(BASE_RESULT_DIR, "figures")

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(LOGS_DIR,    exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# =========================
# PATH CONFIG — CLIENTS
# =========================
DDR_NPZ_PATH       = "/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DDR_dataset_224.npz"
DDR_LABEL_PATH     = "/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading.csv"
EYEPACS_NPZ_PATH   = "/content/drive/MyDrive/S-Class/Orion/OrionFL/EyePACS_Dataset/training/EyePACS_dataset_224.npz"
EYEPACS_LABEL_PATH = "/content/drive/MyDrive/S-Class/Orion/OrionFL/EyePACS_Dataset/training/trainLabels.csv"
APTOS_NPZ_PATH     = "/content/drive/MyDrive/S-Class/Orion/OrionFL/APTOS_2019/preprocessed/orion_dr_224.npz"

# =========================
# TRAINING CONFIG
# =========================
IMG_SIZE               = (224, 224)
BATCH_SIZE             = 32
NUM_CLASSES            = 5

CLIENT_NAMES           = ["DDR", "EyePACS", "APTOS"]

LEARNING_RATE_STAGE1   = 1e-3
LEARNING_RATE_STAGE2   = 1e-6
UNFREEZE_LAST_N_LAYERS = 10
EPOCHS_STAGE1          = 15
EPOCHS_STAGE2          = 15

# =========================
# DEBUG INFO
# =========================
print("TensorFlow :", tf.__version__)
print("MODEL_NAME :", MODEL_NAME)

print()
print("DDR_NPZ     :", os.path.exists(DDR_NPZ_PATH))
print("DDR_LABEL   :", os.path.exists(DDR_LABEL_PATH))
print("EYEPACS_NPZ :", os.path.exists(EYEPACS_NPZ_PATH))
print("EYEPACS_LBL :", os.path.exists(EYEPACS_LABEL_PATH))
print("APTOS_NPZ   :", os.path.exists(APTOS_NPZ_PATH))
print("\nAll imports OK.")

In [ ]:
# =========================
# CELL 2 — LOAD CLIENT 0: DDR
# =========================
ddr_data     = np.load(DDR_NPZ_PATH, allow_pickle=True)
print("DDR NPZ keys:", ddr_data.files)
X_ddr        = ddr_data["X"]
ddr_label_df = pd.read_csv(DDR_LABEL_PATH)
y_ddr        = ddr_label_df["diagnosis"].values.astype(np.int64)
assert len(X_ddr) == len(y_ddr), f"DDR mismatch: {len(X_ddr)} vs {len(y_ddr)}"
print("DDR — X:", X_ddr.shape, "| y:", y_ddr.shape)
print(pd.Series(y_ddr).value_counts().sort_index())

In [ ]:
# =========================
# CELL 3 — LOAD CLIENT 1: EyePACS
# =========================
eyepacs_data     = np.load(EYEPACS_NPZ_PATH, allow_pickle=True)
X_eyepacs        = eyepacs_data["images"]
eyepacs_label_df = pd.read_csv(EYEPACS_LABEL_PATH)
y_eyepacs        = eyepacs_label_df["level"].values.astype(np.int64)
assert len(X_eyepacs) == len(y_eyepacs), f"EyePACS mismatch: {len(X_eyepacs)} vs {len(y_eyepacs)}"
print("EyePACS — X:", X_eyepacs.shape, "| y:", y_eyepacs.shape)
print(pd.Series(y_eyepacs).value_counts().sort_index())

In [ ]:
# =========================
# CELL 4 — LOAD CLIENT 2: APTOS
# =========================
aptos_data = np.load(APTOS_NPZ_PATH, allow_pickle=True)
X_aptos    = aptos_data["images"]
y_aptos    = aptos_data["labels"].astype(np.int64)
assert len(X_aptos) == len(y_aptos), f"APTOS mismatch: {len(X_aptos)} vs {len(y_aptos)}"
print("APTOS — X:", X_aptos.shape, "| y:", y_aptos.shape)
print(pd.Series(y_aptos).value_counts().sort_index())

In [ ]:
# =========================
# CELL 5 — POOL + GLOBAL 70:15:15 SPLIT
# =========================
X_all = np.concatenate([X_ddr, X_eyepacs, X_aptos], axis=0)
y_all = np.concatenate([y_ddr, y_eyepacs, y_aptos], axis=0)
source_all = np.array(
    ["DDR"]     * len(y_ddr) +
    ["EyePACS"] * len(y_eyepacs) +
    ["APTOS"]   * len(y_aptos)
)

print(f"Total pooled: {len(y_all):,}")
print("Label distribution:")
print(pd.Series(y_all).value_counts().sort_index())

X_train, X_temp, y_train, y_temp, src_train, src_temp = train_test_split(
    X_all, y_all, source_all, test_size=0.30, random_state=SEED, stratify=y_all
)
X_val, X_test, y_val, y_test, src_val, src_test = train_test_split(
    X_temp, y_temp, src_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}")

In [ ]:
# =========================
# CELL 6 — PREPROCESS (EfficientNetB0 — float32 0-255, NO preprocess_input)
# =========================
X_train = X_train.astype("float32")
X_val   = X_val.astype("float32")
X_test  = X_test.astype("float32")

print("Preprocessing done (float32, range 0-255).")
print(f"  X_train: {X_train.dtype} | min={X_train[0].min():.1f} max={X_train[0].max():.1f}")

In [ ]:
# =========================
# CELL 7 — CLASS WEIGHTS + TF DATASETS
# =========================
from sklearn.utils.class_weight import compute_class_weight

AUTOTUNE = tf.data.AUTOTUNE

classes      = np.unique(y_train)
cw_array     = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, cw_array)}
print("Class weights:", class_weights)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
], name="light_augmentation")

def augment_fn(x, y):
    return data_augmentation(x, training=True), y

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train), seed=SEED).batch(BATCH_SIZE)
    .map(augment_fn, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))

val_ds  = tf.data.Dataset.from_tensor_slices((X_val,  y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print(f"train_ds: {len(train_ds)} batches | val_ds: {len(val_ds)} | test_ds: {len(test_ds)}")

In [ ]:
# =========================
# CELL 8 — BUILD EfficientNetB0 MODEL
# =========================
def build_efficientnetb0_model(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    base_model.trainable = False

    inputs  = keras.Input(shape=input_shape)
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.4)(x)
    outputs = layers.Dense(
        num_classes, activation="softmax",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

print("EfficientNetB0 model builder ready.")

In [ ]:
# =========================
# CELL 9 — BUILD MODEL
# =========================
model, base_model = build_efficientnetb0_model(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    num_classes=NUM_CLASSES
)
model.summary()

In [ ]:
# =========================
# CELL 10 — COMPILE + TRAIN STAGE 1 (Frozen Backbone)
# =========================
STAGE1_BEST_PATH = os.path.join(MODELS_DIR, "best_efficientnetb0_stage1.keras")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_STAGE1),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks_stage1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                       patience=2, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(filepath=STAGE1_BEST_PATH,
                                     monitor="val_loss", save_best_only=True, verbose=1)
]

history_stage1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_STAGE1, class_weight=class_weights,
    callbacks=callbacks_stage1, verbose=1
)

In [ ]:
# =========================
# CELL 11 — UNFREEZE + STAGE 2 (Fine-Tuning)
# =========================
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAST_N_LAYERS]:
    layer.trainable = False
for layer in base_model.layers[-UNFREEZE_LAST_N_LAYERS:]:
    layer.trainable = True
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable layers in backbone: {trainable} / {len(base_model.layers)}")

In [ ]:
# =========================
# CELL 12 — COMPILE + TRAIN STAGE 2
# =========================
STAGE2_BEST_PATH = os.path.join(MODELS_DIR, "best_efficientnetb0_finetune.keras")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_STAGE2),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks_stage2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                       patience=2, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(filepath=STAGE2_BEST_PATH,
                                     monitor="val_loss", save_best_only=True, verbose=1)
]

history_stage2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_STAGE2, class_weight=class_weights,
    callbacks=callbacks_stage2, verbose=1
)

In [ ]:
# =========================
# CELL 13 — SAVE FINAL MODEL
# =========================
FINAL_MODEL_PATH = os.path.join(MODELS_DIR, "efficientnetb0_multiclient_final.keras")
model.save(FINAL_MODEL_PATH)
print("Model saved:", FINAL_MODEL_PATH)

In [ ]:
# =========================
# CELL 14 — QUICK SUMMARY
# =========================
s1_acc  = max(history_stage1.history.get("val_accuracy", [None]))
s2_acc  = max(history_stage2.history.get("val_accuracy", [None]))
f_t_acc = history_stage2.history.get("accuracy",     [None])[-1]
f_v_acc = history_stage2.history.get("val_accuracy", [None])[-1]

print("=" * 55)
print("EFFICIENTNETB0 CENTRALIZED BASELINE")
print("=" * 55)
print(f"Best Stage 1 Val Acc : {s1_acc:.4f}" if s1_acc else "")
print(f"Best Stage 2 Val Acc : {s2_acc:.4f}" if s2_acc else "")
print(f"Final Train Acc      : {f_t_acc:.4f}" if f_t_acc else "")
print(f"Final Val Acc        : {f_v_acc:.4f}" if f_v_acc else "")
print("=" * 55)

In [ ]:
# =========================
# CELL 15 — LEARNING BEHAVIOUR TABLE
# =========================
learning_behaviour_df = pd.DataFrame({
    "epoch"        : list(range(1, len(history_stage1.history["accuracy"]) +
                                   len(history_stage2.history["accuracy"]) + 1)),
    "train_accuracy": history_stage1.history["accuracy"]     + history_stage2.history["accuracy"],
    "val_accuracy"  : history_stage1.history["val_accuracy"] + history_stage2.history["val_accuracy"],
    "train_loss"    : history_stage1.history["loss"]         + history_stage2.history["loss"],
    "val_loss"      : history_stage1.history["val_loss"]     + history_stage2.history["val_loss"],
    "stage"         : (["stage1"] * len(history_stage1.history["accuracy"]) +
                       ["stage2"] * len(history_stage2.history["accuracy"]))
})
display(learning_behaviour_df.head())
learning_behaviour_df.to_csv(os.path.join(LOGS_DIR, "learning_behaviour_table.csv"), index=False)
print("Saved: learning_behaviour_table.csv")

In [ ]:
# =========================
# CELL 16 — ACCURACY / LOSS PLOT
# =========================
stage1_len = len(history_stage1.history["accuracy"])
epochs_all = list(range(1, len(learning_behaviour_df) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_all, learning_behaviour_df["train_accuracy"], label="Train")
axes[0].plot(epochs_all, learning_behaviour_df["val_accuracy"],   label="Val")
axes[0].axvline(x=stage1_len, color="gray", linestyle="--", linewidth=1, label="Stage 1→2")
axes[0].set_title("EfficientNetB0 — Accuracy\nCentralized Pooled Baseline")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()

axes[1].plot(epochs_all, learning_behaviour_df["train_loss"],   label="Train", color="orange")
axes[1].plot(epochs_all, learning_behaviour_df["val_loss"],     label="Val",   color="red")
axes[1].axvline(x=stage1_len, color="gray", linestyle="--", linewidth=1, label="Stage 1→2")
axes[1].set_title("EfficientNetB0 — Loss\nCentralized Pooled Baseline")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()

plt.suptitle("MultiClient Centralized — EfficientNetB0 | DDR + EyePACS + APTOS", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "baseline_train_curves.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: baseline_train_curves.png")

In [ ]:
# =========================
# CELL 17 — BUILD PREDICTION OBJECTS
# =========================
test_probs = model.predict(test_ds, verbose=1)
test_preds = np.argmax(test_probs, axis=1)

y_true = np.array(y_test)
y_pred = np.array(test_preds)
y_prob = np.array(test_probs)
print("y_true:", y_true.shape, "| y_pred:", y_pred.shape, "| y_prob:", y_prob.shape)

In [ ]:
# =========================
# CELL 18 — MAIN SUMMARY TABLE
# =========================
acc = accuracy_score(y_true, y_pred)
precision_macro,    recall_macro,    f1_macro,    _ = precision_recall_fscore_support(y_true, y_pred, average="macro",    zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

y_true_bin       = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
auc_macro_ovr    = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="macro")
auc_weighted_ovr = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="weighted")

summary_df = pd.DataFrame([{
    "experiment_name"    : EXPERIMENT_NAME,
    "dataset_name"       : DATASET_NAME,
    "model_name"         : MODEL_NAME,
    "setup"              : "Centralized Pooled",
    "accuracy"           : acc,
    "precision_macro"    : precision_macro,
    "recall_macro"       : recall_macro,
    "f1_macro"           : f1_macro,
    "precision_weighted" : precision_weighted,
    "recall_weighted"    : recall_weighted,
    "f1_weighted"        : f1_weighted,
    "auc_macro_ovr"      : auc_macro_ovr,
    "auc_weighted_ovr"   : auc_weighted_ovr,
}])
display(summary_df)
summary_df.to_csv( os.path.join(LOGS_DIR, "baseline_summary_table.csv"),  index=False)
summary_df.to_json(os.path.join(LOGS_DIR, "baseline_summary_table.json"), orient="records", indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 19 — CLASSIFICATION REPORT
# =========================
report_text = classification_report(y_true, y_pred, digits=4)
report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)
print(report_text)
with open(os.path.join(LOGS_DIR, "classification_report_test.txt"),  "w") as f: f.write(report_text)
with open(os.path.join(LOGS_DIR, "classification_report_test.json"), "w") as f: json.dump(report_dict, f, indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 20 — PER-CLASS METRICS
# =========================
precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), zero_division=0
)
per_class_df = pd.DataFrame({
    "class"    : np.arange(NUM_CLASSES),
    "precision": precision_cls,
    "recall"   : recall_cls,
    "f1_score" : f1_cls,
    "support"  : support_cls
})
display(per_class_df)
per_class_df.to_csv( os.path.join(LOGS_DIR, "per_class_metrics.csv"),  index=False)
per_class_df.to_json(os.path.join(LOGS_DIR, "per_class_metrics.json"), orient="records", indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 21 — ROC CURVE + AUC
# =========================
y_true_bin   = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
roc_auc_dict = {}
plt.figure(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    auc_val     = auc(fpr, tpr)
    roc_auc_dict[f"class_{i}"] = float(auc_val)
    plt.plot(fpr, tpr, label=f"Class {i} (AUC={auc_val:.4f})")
plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"ROC Curve — EfficientNetB0 Centralized Pooled\nDDR + EyePACS + APTOS")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "roc_curve_baseline.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()
with open(os.path.join(LOGS_DIR, "roc_auc_per_class.json"), "w") as f:
    json.dump(roc_auc_dict, f, indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 22 — CONFUSION MATRIX
# =========================
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=np.arange(NUM_CLASSES)).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix — EfficientNetB0 Centralized Pooled")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "confusion_matrix_heatmap_baseline.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

cm_df = pd.DataFrame(cm,
    index=[f"true_{i}" for i in range(NUM_CLASSES)],
    columns=[f"pred_{i}" for i in range(NUM_CLASSES)])
cm_df.to_csv(os.path.join(LOGS_DIR, "confusion_matrix_table.csv"))
with open(os.path.join(LOGS_DIR, "confusion_matrix_values_test.json"), "w") as f:
    json.dump({"confusion_matrix": cm.tolist()}, f, indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 23 — PER-DATASET TEST ANALYSIS
# =========================
print("Per-Dataset Performance on Test Set")
print("=" * 55)

per_dataset_results = []
for dataset_name in ["APTOS", "DDR", "EyePACS"]:
    mask = src_test == dataset_name
    if mask.sum() == 0:
        print(f"{dataset_name}: no samples"); continue

    y_t  = y_true[mask]; y_p  = y_pred[mask]
    y_b  = label_binarize(y_t, classes=np.arange(NUM_CLASSES))
    y_pr = y_prob[mask]

    ds_acc          = accuracy_score(y_t, y_p)
    ds_p, ds_r, ds_f1, _ = precision_recall_fscore_support(y_t, y_p, average="macro", zero_division=0)
    try:    ds_auc = roc_auc_score(y_b, y_pr, multi_class="ovr", average="macro")
    except: ds_auc = None

    auc_str = f"{ds_auc:.4f}" if ds_auc is not None else "N/A"
    per_dataset_results.append({
        "dataset": dataset_name, "n_test_samples": int(mask.sum()),
        "accuracy": round(ds_acc, 4), "precision_macro": round(ds_p, 4),
        "recall_macro": round(ds_r, 4), "f1_macro": round(ds_f1, 4),
        "auc_macro_ovr": round(ds_auc, 4) if ds_auc is not None else None,
    })
    print(f"  {dataset_name:<10} | n={mask.sum():>5,} | Acc={ds_acc:.4f} | F1={ds_f1:.4f} | AUC={auc_str}")

per_dataset_df = pd.DataFrame(per_dataset_results)
display(per_dataset_df)
per_dataset_df.to_csv( os.path.join(LOGS_DIR, "per_dataset_test_metrics.csv"),  index=False)
per_dataset_df.to_json(os.path.join(LOGS_DIR, "per_dataset_test_metrics.json"), orient="records", indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 24 — COMPARISON READY JSON
# =========================
comparison_row = {
    "method"           : "Centralized_Pooled",
    "model"            : MODEL_NAME,
    "dataset"          : DATASET_NAME,
    "clients"          : CLIENT_NAMES,
    "accuracy"         : float(acc),
    "precision_macro"  : float(precision_macro),
    "recall_macro"     : float(recall_macro),
    "f1_macro"         : float(f1_macro),
    "auc_macro_ovr"    : float(auc_macro_ovr),
    "notes"            : "EfficientNetB0, Centralized Pooled, MultiClient (DDR+EyePACS+APTOS)."
}
with open(os.path.join(LOGS_DIR, "comparison_ready_baseline.json"), "w") as f:
    json.dump(comparison_row, f, indent=4)
print("Saved: comparison_ready_baseline.json")
print(json.dumps(comparison_row, indent=4))